라이브러리 로드

In [1]:
# Import or install Sionna
import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

import importlib
import utils as js
js = importlib.reload(js)

no_preview = False # Toggle to False to use the preview widget

2026-02-25 18:53:56.931432: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-25 18:53:56.944887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772013236.959112 2918206 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772013236.963402 2918206 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772013236.974500 2918206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

1. Scene XML 파일 로드 및 재질 확인

In [4]:
# XML 파일 경로 보정 및 로드
xml_path = "/home/mh/kmh/sionna-rt/src/sionna/rt/scenes/etri/etri_260225_fixed.xml"
scene_dir = os.path.dirname(xml_path)

scene = load_scene(xml_path, merge_shapes=False)
scene.frequency = 28e9
# 재질(Material) 정보 확인
js.print_scene_materials(scene)


Object Name                    | Assigned Material   
elm__2                         | itu_concrete        
elm__3                         | itu_concrete        
elm__5                         | itu_wood            
elm__7                         | itu_concrete        
elm__9                         | itu_concrete        
elm__11                        | itu_concrete        
elm__13                        | itu_concrete        

[정의된 재질 목록]
 - 이름: itu_concrete    (Type: concrete, Thickness: [0.1])
 - 이름: itu_wood        (Type: wood, Thickness: [0.1])


2. 도로 재질 변경

In [5]:
red_road_mat = ITURadioMaterial(name="red_road_mat_7",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

road_object_id = "elm__7"

if road_object_id in scene.objects:
    scene.objects[road_object_id].radio_material = red_road_mat
    print(f"[설정] 도로({road_object_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__7)를 빨간색으로 변경했습니다.


3. 도로 정점 추출

In [6]:
road_object_id = "elm__7"
road_positions = js.get_road_positions_from_object(
    scene,
    road_object_id,
    dtype=np.float16,       # 한글 주석: 연산용 기본 타입
    round_decimals=3,       # 한글 주석: 화면 표시 자릿수만 줄임
)

if len(road_positions) == 0:
    print("[안내] 추출된 좌표가 없습니다.")
else:
    print("dtype:", road_positions.dtype)
    print("first road position:", road_positions[0])

[탐색] Mitsuba Scene 내부에서 'elm__7' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 308개 (dtype=float16)
dtype: float16
first road position: [-279.8   81.2    0.3]


4. 초기 위치 테스트

In [7]:
# 설정 및 초기화
target_pos = road_positions[0]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[2.294, -279.590, 11]]#, [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1"]#, "Tx_2", "Tx_3"]

# 기존 객체 제거
for name in tx_names + ['Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 시각화
cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: [-279.8   81.2    0.3]
 -> Tx_1 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...
[시각화] 3D 뷰어 실행


***메인 시뮬레이션***

경로 준비

In [8]:
# 한글 주석: 먼저 XY에서 정점 인덱스 분포를 확인
js.plot_vertices_index(
    road_positions,
    width=1000,
    height=800,
    marker_size=5,
)

In [9]:
# 한글 주석: 위 Cell 에서 본 인덱스를 직접 입력
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [0,1,4,6,8,10,12,14,16,18,19,22]

path_data = js.prepare_vehicle_path(
    road_positions=road_positions,
    path_indices=path_indices,
    z_offset=2.7,
    speed_ms=90.0 / 3.6,
    delta_t=0.5,
)

waypoints = path_data["waypoints"]
time_steps = path_data["time_steps"]
print("waypoints:", waypoints.shape, "| time_steps:", len(time_steps))

waypoints: (12, 3) | time_steps: 53


In [ ]:
js.plot_vertices_xy(
    road_positions=road_positions,
    waypoints=waypoints,
    start_point=path_indices[0],
    end_point=path_indices[-1],
    figsize=(9, 7),
)

Scene 초기화 (Tx/Rx + Solver)

In [ ]:
tx_names, rx, solver = js.setup_scene_for_vehicle_path(
    scene=scene,
    path_data=path_data,
    tx_positions=tx_positions,
    tx_names=tx_names,
    tx_power_dbm=43.0,
    display_radius=20.0
)

start_pos, start_vel = js.get_state_at_time(path_data, 0.0)
print("초기 위치:", np.round(start_pos, 3))
print("초기 속도:", np.round(start_vel, 3))

Interactive 위젯 실행

In [ ]:
slider, output_widget = js.create_vehicle_simulation_widgets(
    scene=scene,
    path_data=path_data,
    tx_names=tx_names,
    rx=rx,
    solver=solver,
    max_depth=3,
    max_num_paths_per_src=100000,
    samples_per_src=100000,
    diffuse_reflection=True,
    diffraction=True,
    synthetic_array=True,
    resolution=(800, 600),
)

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

CIR 측정

In [ ]:
# 1. 이전 셀 등에서 미리 만들어둔 경로 데이터를 불러옵니다.
# path_data = prepare_vehicle_path(road_positions, speed_ms=60/3.6, use_centerline=True)

# 2. path_data 안에서 시간축 배열(time_steps)을 꺼내옵니다.
time_steps = path_data["time_steps"]

frame_idx = 0  # 보고 싶은 프레임 인덱스 (원하는 값으로 바꿔)
t = time_steps[frame_idx]

# 3. get_pos_at_time 대신 좀 전에 만든 get_state_at_time을 사용하여 위치와 속도를 동시에 가져옵니다.
current_pos, current_vel = js.get_state_at_time(path_data, t)

# 4. 위치/방향 업데이트 (속도까지 업데이트 해주는 것이 도플러 효과에 좋습니다!)
rx.position = current_pos
rx.velocity = current_vel

for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 5. 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True, synthetic_array=True)

# 6. CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)


CIR 시각화

In [ ]:
# 모양(shape)이 [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths] 인 경우 (5차원)
t = tau[0, 0, :] / 1e-9            # tau는 주로 [rx, tx, path] 형태
# (참고: 버전에 따라 tau도 a와 차원이 같을 수 있으니 안 맞으면 tau[0, 0, 0, 0, :] 로 수정)
a_abs = np.abs(a)[0, 0, 0, 0, :, 0]

# --- 그래프 그리기 ---
plt.figure(figsize=(8, 4))
plt.title("Channel Impulse Response")
plt.stem(t, a_abs, basefmt=" ")
plt.xlim(left=0)
plt.xlabel(r"$\tau$ [ns]")
plt.ylabel(r"$|a|$")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

TAP 시각화

In [ ]:
# 한글 주석: taps 설정값
bandwidth = 500e6          # 채널 대역폭 [Hz]
l_min, l_max = 0, 1500     # 탭 인덱스 범위
sampling_frequency = 10e4         # 시간축 샘플링 주파수 [Hz]
num_time_steps = 16       # 시간 샘플 수(도플러 반영)

In [ ]:
# 함수 한번만 호출하면 관련된 UI와 위젯 이벤트가 예쁘게 반환됩니다.
widget_view = js.create_taps_pdp_widgets(
    scene=scene, 
    rx=rx, 
    solver=solver, 
    path_data=path_data, 
    tx_names=tx_names,
    bandwidth=bandwidth,           # 예: 100e6
    l_min=l_min,                   # 예: -10
    l_max=l_max,                   # 예: 50
    sampling_frequency=sampling_frequency,  # 예: 1e9
    num_time_steps=num_time_steps  # 예: 1
)

# 최종적으로 화면에 표시
display(widget_view)